# Lightweight GenAI And RAG Demo

This notebook supports Session 2 of the training.

## Objectives
- Load a small set of technical energy documents
- Build a simple retrieval workflow
- Find the most relevant documents for a user question
- Assemble a grounded prompt that can be sent to an LLM

This is intentionally lightweight. The goal is to help participants understand the logic of retrieval-augmented generation without needing a full application stack.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

candidate_dirs = [
    Path("../../data/technical_docs"),
    Path("training/data/technical_docs"),
    Path("data/technical_docs"),
]

docs_dir = next((path for path in candidate_dirs if path.exists()), None)
if docs_dir is None:
    raise FileNotFoundError("Could not locate the technical_docs directory from this notebook.")

documents = []
for path in sorted(docs_dir.glob("*.md")):
    documents.append({"source": path.name, "text": path.read_text()})

docs_df = pd.DataFrame(documents)
docs_df

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english")
doc_matrix = vectorizer.fit_transform(docs_df["text"])

def search_docs(query, top_k=3):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, doc_matrix).ravel()
    result = docs_df.copy()
    result["score"] = scores
    return result.sort_values("score", ascending=False).head(top_k).reset_index(drop=True)

query = "Which wells may need surveillance because of rising water cut or poor uptime?"
search_docs(query)

In [ ]:
def build_grounded_prompt(query, top_k=3):
    matches = search_docs(query, top_k=top_k)
    context_blocks = []
    for _, row in matches.iterrows():
        snippet = row["text"][:900].strip()
        context_blocks.append(f"SOURCE: {row['source']}\n{snippet}")

    prompt = f"""You are helping an energy-industry technical team answer a question using retrieved context.

Question:
{query}

Retrieved context:
{'\n\n'.join(context_blocks)}

Instructions:
- Answer only from the retrieved context.
- If the answer is uncertain, say so.
- Cite the source names you used.
- Give a practical, engineer-friendly response.
"""
    return prompt, matches

prompt_text, matches = build_grounded_prompt(
    "What are the likely drivers of underperformance and what follow-up actions should an engineer consider?",
    top_k=3,
)

print(prompt_text)
matches[["source", "score"]]

## How To Use This In The Training

1. Start with a user question.
2. Retrieve the most relevant technical notes.
3. Show the retrieved sources.
4. Build the grounded prompt.
5. Paste that prompt into your preferred LLM to generate a cited answer.

## Suggested Questions
- Which reports mention vibration or dysfunction risk?
- What operational themes appear across the surveillance and offset notes?
- What should a technical chatbot say if the user asks about storage-site screening risk?

## Extension Ideas
- Replace TF-IDF retrieval with embeddings if your environment supports it.
- Add metadata filters such as document type or discipline.
- Wrap the retrieval function into a small app or chatbot prototype.
- Connect the retriever to a tool-enabled agent in a future iteration.